In [20]:
import pandas as pd

df = pd.read_csv("DataCo_Cleaned_Interim.csv")

print("Total rows:", len(df))
print("Unique orders:", df["Order Id"].nunique())

Total rows: 36957
Unique orders: 20652


In [21]:
print("Missing Order IDs:", df["Order Id"].isna().sum())
print("Missing Order Dates:", df["order date (DateOrders)"].isna().sum())
print("Missing Shipping Dates:", df["shipping date (DateOrders)"].isna().sum())

Missing Order IDs: 0
Missing Order Dates: 0
Missing Shipping Dates: 1


In [22]:
orders = df.groupby("Order Id", as_index=False).agg({
    "Order Status": "first",
    "Delivery Status": "first"
})

print("Distinct orders:", len(orders))

Distinct orders: 20652


In [23]:
status_check = df.groupby("Order Id").agg({
    "Order Status": "nunique",
    "Delivery Status": "nunique"
})

print("Inconsistent Order Status:",
      (status_check["Order Status"] > 1).sum())

print("Inconsistent Delivery Status:",
      (status_check["Delivery Status"] > 1).sum())


Inconsistent Order Status: 0
Inconsistent Delivery Status: 0


In [24]:
on_time_statuses = ["Advance shipping", "Shipping on time"]
valid_statuses = on_time_statuses + ["Late delivery"]

orders["On-time Numerator Flag"] = (
    orders["Delivery Status"].isin(on_time_statuses).astype(int)
)

orders["On-time Denominator Flag"] = (
    orders["Delivery Status"].isin(valid_statuses).astype(int)
)

on_time_num = orders["On-time Numerator Flag"].sum()
on_time_den = orders["On-time Denominator Flag"].sum()

on_time_rate = on_time_num / on_time_den * 100 if on_time_den else 0

print("On-time numerator:", on_time_num)
print("On-time denominator:", on_time_den)
print("On-time delivery rate:", round(on_time_rate, 2), "%")

On-time numerator: 8547
On-time denominator: 19979
On-time delivery rate: 42.78 %


In [25]:
orders["Cancellation Numerator Flag"] = (
    orders["Order Status"].eq("CANCELED").fillna(False).astype(int)
)

orders["Cancellation Denominator Flag"] = 1

cancel_num = orders["Cancellation Numerator Flag"].sum()
cancel_den = orders["Cancellation Denominator Flag"].sum()

cancel_rate = cancel_num / cancel_den * 100 if cancel_den else 0

print("Cancellation numerator:", cancel_num)
print("Cancellation denominator:", cancel_den)
print("Cancellation rate:", round(cancel_rate, 2), "%")



Cancellation numerator: 314
Cancellation denominator: 20652
Cancellation rate: 1.52 %


In [26]:
sample = orders.sample(n=min(23, len(orders)), random_state=42).copy()

sample["Expected On-time"] = (
    sample["Delivery Status"].isin(on_time_statuses).astype(int)
)

sample["Expected Cancellation"] = (
    sample["Order Status"].eq("CANCELED").fillna(False).astype(int)
)

sample["On-time Check"] = (
    sample["On-time Numerator Flag"] == sample["Expected On-time"]
)

sample["Cancellation Check"] = (
    sample["Cancellation Numerator Flag"] == sample["Expected Cancellation"]
)

display(sample[[
    "Order Id", "Order Status", "Delivery Status",
    "On-time Numerator Flag", "Expected On-time", "On-time Check",
    "Cancellation Numerator Flag", "Expected Cancellation",
    "Cancellation Check"
]])

,Order Id,Order Status,Delivery Status,On-time Numerator Flag,Expected On-time,On-time Check,Cancellation Numerator Flag,Expected Cancellation,Cancellation Check
2408,8842,CLOSED,Advance shipping,1,1,True,0,0,True
8213,29399,PAYMENT_REVIEW,Late delivery,0,0,True,0,0,True
341,1157,PENDING_PAYMENT,Late delivery,0,0,True,0,0,True
12382,45242,COMPLETE,Advance shipping,1,1,True,0,0,True
16010,59103,COMPLETE,Shipping on time,1,1,True,0,0,True
18819,70252,COMPLETE,Advance shipping,1,1,True,0,0,True
3115,11348,COMPLETE,Shipping on time,1,1,True,0,0,True
3024,10972,CLOSED,Late delivery,0,0,True,0,0,True
3248,11811,COMPLETE,Advance shipping,1,1,True,0,0,True
4921,17724,COMPLETE,Late delivery,0,0,True,0,0,True


In [27]:
expected = {
    "Rows": 180519,
    "Orders": 65752,
    "On-time numerator": 26849,
    "On-time denominator": 62897,
    "Cancellation numerator": 1367,
    "Cancellation denominator": 65752
}

actual = {
    "Rows": len(df),
    "Orders": len(orders),
    "On-time numerator": int(on_time_num),
    "On-time denominator": int(on_time_den),
    "Cancellation numerator": int(cancel_num),
    "Cancellation denominator": int(cancel_den)
}

for name in expected:
    print(name, "| Actual:", actual[name],
          "| Expected:", expected[name],
          "| Match:", actual[name] == expected[name])

Rows | Actual: 36957 | Expected: 180519 | Match: False
Orders | Actual: 20652 | Expected: 65752 | Match: False
On-time numerator | Actual: 8547 | Expected: 26849 | Match: False
On-time denominator | Actual: 19979 | Expected: 62897 | Match: False
Cancellation numerator | Actual: 314 | Expected: 1367 | Match: False
Cancellation denominator | Actual: 20652 | Expected: 65752 | Match: False


In [28]:
print("\nFINAL VALIDATION SUMMARY")
print("On-time delivery rate:", round(on_time_rate, 2), "%")
print("Cancellation rate:", round(cancel_rate, 2), "%")
print("All sample checks passed:",
      sample["On-time Check"].all()
      and sample["Cancellation Check"].all())


FINAL VALIDATION SUMMARY
On-time delivery rate: 42.78 %
Cancellation rate: 1.52 %
All sample checks passed: True
